# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliza1800/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule idea

I will prioritize pages using two observable signals: **days since last update** and **search impressions**.

Pages that have not been updated for longer and still receive meaningful search impressions are stronger candidates for a refresh-oriented action. The score is used for prioritization and decision-support, not as a prediction of future performance.

### Reason codes

* **STALE_HIGH_VALUE** — the page has not been updated for a longer period and has meaningful search visibility.
* **VISIBLE_NOT_STALE** — the page has meaningful search visibility but is not strongly stale.
* **LOW_SIGNAL** — the page does not have a strong combination of the two signals.


In [20]:
# Signal 1: days since last update
# Bucket table with n

stale_bins = [-1, 30, 90, 180, 365, float("inf")]
stale_labels = ["0-30", "31-90", "91-180", "181-365", "366+"]

stale_check = (
    df.assign(
        stale_bucket=pd.cut(
            df["days_since_last_update"],
            bins=stale_bins,
            labels=stale_labels
        )
    )
    .groupby("stale_bucket", observed=False)
    .size()
    .reset_index(name="n")
)

print("Signal: days_since_last_update")
display(stale_check)

Signal: days_since_last_update


,stale_bucket,n
0,0-30,20480
1,31-90,175
2,91-180,9171
3,181-365,169
4,366+,5


### Signal 1 verdict

**Verdict: MIXED**

The observed distribution shows meaningful variation in update age, but most pages are concentrated in the 0–30 and 91–180 day buckets. Very old pages are rare, with only 5 pages in the 366+ day bucket. This makes `days_since_last_update` useful as a directional staleness signal, but it should not be treated as a strong standalone indicator.


In [21]:
# Signal 2: search impressions
# Bucket table with n

impression_bins = [-1, 100, 500, 1000, 5000, float("inf")]
impression_labels = ["0-100", "101-500", "501-1000", "1001-5000", "5001+"]

impression_check = (
    df.assign(
        impression_bucket=pd.cut(
            df["impressions_90d"],
            bins=impression_bins,
            labels=impression_labels
        )
    )
    .groupby("impression_bucket", observed=False)
    .size()
    .reset_index(name="n")
)

print("Signal: impressions_90d")
display(impression_check)

Signal: impressions_90d


,impression_bucket,n
0,0-100,8006
1,101-500,5279
2,501-1000,3206
3,1001-5000,7359
4,5001+,6150


### Signal 2 verdict

**Verdict: CONFIRMED**

The observed distribution shows substantial variation in search impressions across the five buckets. There are meaningful numbers of pages at both lower and higher visibility levels, so `impressions_90d` provides a usable signal for prioritization.


### Signal conclusion

The two signals provide different types of information. `days_since_last_update` provides a directional staleness signal, while `impressions_90d` provides a stronger and more evenly distributed search-visibility signal.

Because the staleness signal is MIXED rather than strongly confirmed, the baseline will use it as one component of the score rather than treating it as proof that a page needs a refresh. Search impressions will provide the second component for prioritization.

The rule is intended for decision-support using observed data only. It does not use future outcomes or target labels as inputs.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline scoring rule

I will combine the two checked signals into one simple priority score.

`days_since_last_update` contributes a staleness score, while `impressions_90d` contributes a search-visibility score. Higher values on either signal increase the priority score.

The final score is used to rank pages for decision-support. It is not a prediction of future performance.

The rule assigns one reason code, **BASELINE_PRIORITY**, to every ranked page and uses the action label **Review for refresh** for higher-priority pages and **Monitor** for lower-priority pages.


In [22]:
# STEP 2 — Build the baseline score

import numpy as np
import pandas as pd
from pathlib import Path

baseline = df.copy()

# 1. Staleness score
# More days since last update = higher score
baseline["staleness_score"] = np.select(
    [
        baseline["days_since_last_update"] >= 180,
        baseline["days_since_last_update"] >= 90,
        baseline["days_since_last_update"] >= 30
    ],
    [3, 2, 1],
    default=0
)

# 2. Search visibility score
# More impressions = higher score
baseline["visibility_score"] = np.select(
    [
        baseline["impressions_90d"] >= 5000,
        baseline["impressions_90d"] >= 1000,
        baseline["impressions_90d"] >= 500
    ],
    [3, 2, 1],
    default=0
)

# 3. Final score
baseline["score"] = (
    baseline["staleness_score"]
    + baseline["visibility_score"]
)

# 4. ONE reason code
baseline["reason_code"] = "BASELINE_PRIORITY"

# 5. Action label
baseline["action"] = np.where(
    baseline["score"] >= 4,
    "Review for refresh",
    "Monitor"
)

print("Score distribution:")
print(baseline["score"].value_counts().sort_index())

Score distribution:
score
0    10417
1     2166
2     7175
3     4736
4     2832
5     2668
6        6
Name: count, dtype: int64


In [23]:
# Rank all pages from highest to lowest score

baseline = baseline.sort_values(
    ["score", "impressions_90d", "days_since_last_update"],
    ascending=[False, False, False]
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

print("Total ranked pages:", len(baseline))

display(
    baseline[
        [
            "rank",
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "staleness_score",
            "visibility_score",
            "score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

Total ranked pages: 30000


,rank,content_id,days_since_last_update,impressions_90d,staleness_score,visibility_score,score,reason_code,action
0,1,content_cf56e2e2e282,194,61678,3,3,6,BASELINE_PRIORITY,Review for refresh
1,2,content_7368877ea310,194,59472,3,3,6,BASELINE_PRIORITY,Review for refresh
2,3,content_1bfaa38ff26c,194,25715,3,3,6,BASELINE_PRIORITY,Review for refresh
3,4,content_0a91db491d14,193,13299,3,3,6,BASELINE_PRIORITY,Review for refresh
4,5,content_5feee3994adb,194,7812,3,3,6,BASELINE_PRIORITY,Review for refresh
5,6,content_c2d929d83eaa,193,7558,3,3,6,BASELINE_PRIORITY,Review for refresh
6,7,content_5fe46e04994d,104,517715,2,3,5,BASELINE_PRIORITY,Review for refresh
7,8,content_2dba2b1f9536,104,443434,2,3,5,BASELINE_PRIORITY,Review for refresh
8,9,content_2c2606c5d176,104,347399,2,3,5,BASELINE_PRIORITY,Review for refresh
9,10,content_cb112fce36be,104,309910,2,3,5,BASELINE_PRIORITY,Review for refresh


In [24]:
# Write the ranked queue to the required output path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

output_columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action"
]

ranked_queue = baseline[output_columns].copy()

ranked_queue.to_csv(
    output_path,
    index=False
)

print("CSV successfully written:")
print(output_path)

print("\nRows written:", len(ranked_queue))

display(ranked_queue.head(10))

CSV successfully written:
work/outputs/baseline_action_score.csv

Rows written: 30000


,rank,content_id,score,reason_code,action
0,1,content_cf56e2e2e282,6,BASELINE_PRIORITY,Review for refresh
1,2,content_7368877ea310,6,BASELINE_PRIORITY,Review for refresh
2,3,content_1bfaa38ff26c,6,BASELINE_PRIORITY,Review for refresh
3,4,content_0a91db491d14,6,BASELINE_PRIORITY,Review for refresh
4,5,content_5feee3994adb,6,BASELINE_PRIORITY,Review for refresh
5,6,content_c2d929d83eaa,6,BASELINE_PRIORITY,Review for refresh
6,7,content_5fe46e04994d,5,BASELINE_PRIORITY,Review for refresh
7,8,content_2dba2b1f9536,5,BASELINE_PRIORITY,Review for refresh
8,9,content_2c2606c5d176,5,BASELINE_PRIORITY,Review for refresh
9,10,content_cb112fce36be,5,BASELINE_PRIORITY,Review for refresh


In [25]:
# Verify that the required CSV exists and has the expected structure

import os

print("File exists:", os.path.exists("work/outputs/baseline_action_score.csv"))

check = pd.read_csv("work/outputs/baseline_action_score.csv")

print("CSV shape:", check.shape)
print("CSV columns:", check.columns.tolist())

display(check.head(10))

File exists: True
CSV shape: (30000, 5)
CSV columns: ['rank', 'content_id', 'score', 'reason_code', 'action']


,rank,content_id,score,reason_code,action
0,1,content_cf56e2e2e282,6,BASELINE_PRIORITY,Review for refresh
1,2,content_7368877ea310,6,BASELINE_PRIORITY,Review for refresh
2,3,content_1bfaa38ff26c,6,BASELINE_PRIORITY,Review for refresh
3,4,content_0a91db491d14,6,BASELINE_PRIORITY,Review for refresh
4,5,content_5feee3994adb,6,BASELINE_PRIORITY,Review for refresh
5,6,content_c2d929d83eaa,6,BASELINE_PRIORITY,Review for refresh
6,7,content_5fe46e04994d,5,BASELINE_PRIORITY,Review for refresh
7,8,content_2dba2b1f9536,5,BASELINE_PRIORITY,Review for refresh
8,9,content_2c2606c5d176,5,BASELINE_PRIORITY,Review for refresh
9,10,content_cb112fce36be,5,BASELINE_PRIORITY,Review for refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

The top 20 pages are reviewed as decision-support recommendations rather than automatic refresh decisions.

For each page, I consider the observed score, staleness, and search visibility. I also record why the page was prioritized, how confident I am in the recommendation, and what additional information could make the recommendation wrong.

The review is based only on observed dataset signals and does not use future outcomes or target labels.

In [26]:
top20 = baseline.head(20).copy()

top20_review = top20[
    [
        "rank",
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "staleness_score",
        "visibility_score",
        "score",
        "reason_code",
        "action"
    ]
].copy()

top20_review["confidence_note"] = np.where(
    top20_review["score"] >= 6,
    "High relative to this baseline because both signals are at their strongest levels.",
    "Moderate because the priority is based on the observed baseline signals."
)

top20_review["what_would_make_it_wrong"] = (
    "Recent update activity, changing business priorities, or low-quality search visibility could reduce the need for refresh."
)

display(top20_review)

,rank,content_id,days_since_last_update,impressions_90d,staleness_score,visibility_score,score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,194,61678,3,3,6,BASELINE_PRIORITY,Review for refresh,High relative to this baseline because both si...,"Recent update activity, changing business prio..."
1,2,content_7368877ea310,194,59472,3,3,6,BASELINE_PRIORITY,Review for refresh,High relative to this baseline because both si...,"Recent update activity, changing business prio..."
2,3,content_1bfaa38ff26c,194,25715,3,3,6,BASELINE_PRIORITY,Review for refresh,High relative to this baseline because both si...,"Recent update activity, changing business prio..."
3,4,content_0a91db491d14,193,13299,3,3,6,BASELINE_PRIORITY,Review for refresh,High relative to this baseline because both si...,"Recent update activity, changing business prio..."
4,5,content_5feee3994adb,194,7812,3,3,6,BASELINE_PRIORITY,Review for refresh,High relative to this baseline because both si...,"Recent update activity, changing business prio..."
5,6,content_c2d929d83eaa,193,7558,3,3,6,BASELINE_PRIORITY,Review for refresh,High relative to this baseline because both si...,"Recent update activity, changing business prio..."
6,7,content_5fe46e04994d,104,517715,2,3,5,BASELINE_PRIORITY,Review for refresh,Moderate because the priority is based on the ...,"Recent update activity, changing business prio..."
7,8,content_2dba2b1f9536,104,443434,2,3,5,BASELINE_PRIORITY,Review for refresh,Moderate because the priority is based on the ...,"Recent update activity, changing business prio..."
8,9,content_2c2606c5d176,104,347399,2,3,5,BASELINE_PRIORITY,Review for refresh,Moderate because the priority is based on the ...,"Recent update activity, changing business prio..."
9,10,content_cb112fce36be,104,309910,2,3,5,BASELINE_PRIORITY,Review for refresh,Moderate because the priority is based on the ...,"Recent update activity, changing business prio..."


### Top-20 review conclusion

The top 20 recommendations are consistent with the baseline rule. The highest-ranked pages combine stronger staleness and search-visibility signals, while the next group is prioritized mainly because of very high search impressions combined with moderate staleness.

These recommendations should be treated as review candidates rather than automatic refresh decisions. Additional context such as recent editorial activity, business priorities, and content quality could change the final decision.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

I also review weak-ranked pages to check whether the baseline behaves sensibly at the bottom of the queue.

The baseline should prioritize pages using only the two observed signals selected earlier: `days_since_last_update` and `impressions_90d`.

I will also check the input columns for future outcome fields, target labels, and other fields that should not be used as scoring inputs.

In [27]:
weak_picks = baseline.tail(10).copy()

weak_review = weak_picks[
    [
        "rank",
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "score",
        "reason_code",
        "action"
    ]
].copy()

display(weak_review)

,rank,content_id,days_since_last_update,impressions_90d,score,reason_code,action
29990,29991,content_bb600f317035,1,1,0,BASELINE_PRIORITY,Monitor
29991,29992,content_92ceb4aee549,1,1,0,BASELINE_PRIORITY,Monitor
29992,29993,content_994b0a4e4dde,1,1,0,BASELINE_PRIORITY,Monitor
29993,29994,content_5168e96834b9,1,1,0,BASELINE_PRIORITY,Monitor
29994,29995,content_b5fb35404aed,1,1,0,BASELINE_PRIORITY,Monitor
29995,29996,content_8bce3371c63c,1,1,0,BASELINE_PRIORITY,Monitor
29996,29997,content_2a843f006d86,1,1,0,BASELINE_PRIORITY,Monitor
29997,29998,content_1d9eca1ce9cd,1,1,0,BASELINE_PRIORITY,Monitor
29998,29999,content_9ffe1e2e3575,1,1,0,BASELINE_PRIORITY,Monitor
29999,30000,content_0a22a2eeefdd,1,1,0,BASELINE_PRIORITY,Monitor


In [28]:
print("Weak picks action counts:")
print(weak_review["action"].value_counts())

Weak picks action counts:
action
Monitor    10
Name: count, dtype: int64


In [29]:
# Columns actually used by the baseline score
score_inputs = [
    "days_since_last_update",
    "impressions_90d"
]

# Common fields that would be unsafe if used as scoring inputs
leakage_keywords = [
    "label",
    "target",
    "future",
    "outcome",
    "next",
    "trend"
]

possible_leakage = [
    col for col in score_inputs
    if any(keyword in col.lower() for keyword in leakage_keywords)
]

print("Score inputs:", score_inputs)
print("Possible leakage in score inputs:", possible_leakage)

if len(possible_leakage) == 0:
    print("Leakage check: PASS")
else:
    print("Leakage check: REVIEW")

Score inputs: ['days_since_last_update', 'impressions_90d']
Possible leakage in score inputs: []
Leakage check: PASS


### conclusion

The weak picks were assigned the `Monitor` action, which is consistent with their low baseline priority. The leakage check also passed because the baseline score uses only `days_since_last_update` and `impressions_90d` and does not include target, future, outcome, or trend fields.

This confirms that the baseline provides a simple and transparent prioritization rule without using obvious leakage-prone inputs.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.